# Reshaping dataframes

**Learning Objectives:**
- Understand when and why to reshape data
- Use `.transpose()` to swap rows and columns
- Apply `.unstack()` to create columns from index values
- Use `.pivot()` to reshape data with unique combinations
- Apply `.pivot_table()` for reshaping with aggregation
- Use `pd.crosstab()` to create frequency and summary tables
- Convert wide format to long format with `.melt()`

## 1. Introduction to reshaping

📚 **Concept**: Data reshaping is the process of reorganizing how your data is structured. The two main formats are:

- **Wide format**: Each variable has its own column (easier to read)
- **Long format**: Repeated measurements in rows (better for analysis)

**When you need reshaping:**
- Converting data from a database query for visualization
- Preparing data for statistical analysis or modeling
- Creating pivot tables similar to Excel
- Summarizing data from multiple dimensions

In [ ]:
import pandas as pd

## 2. Transpose

📚 **Concept**: Transpose swaps rows and columns in a DataFrame.

💡 **Tip**: This operation is rarely needed in practice, but it's useful when you need to flip the structure of your data.

In [ ]:
# Load the Titanic dataset
titanic = pd.read_csv("titanic.csv")

In [ ]:
# View the first few rows
titanic.head(3)

### 2.1 Basic transpose operation

In [ ]:
# Using the .T attribute (shorthand)
titanic.head(5).T

In [ ]:
# Using .transpose() method (explicit)
titanic.head(5).transpose()

⚠️ **Warning**: Transposing changes all column data types to `object` if columns have mixed types. Avoid reassigning unless necessary.

## 3. Unstack

📚 **Concept**: Unstack creates new columns based on unique values from an index level (usually the inner index).

**When to use**: After grouping data, when you want to reshape from long to wide format.

### 3.1 Unstack with groupby

In [ ]:
# Calculate survival rate by sex and passenger class
survival_rate = titanic.groupby(["sex", "pclass"])["survived"].mean()
survival_rate

In [ ]:
# Unstack to create columns from pclass (inner index)
survival_rate.unstack()

✅ **Key Takeaway**: Unstack transforms the innermost index level into columns, making the data wider and easier to read.

### 3.2 Combining with transpose

💡 **Tip**: You can chain `.T` after `.unstack()` to flip the result.

In [ ]:
# Transpose to have pclass as rows and sex as columns
survival_rate.unstack().T

## 4. Pivot

📚 **Concept**: Pivot creates new columns based on unique values from an existing column.

**When to use**: When you have unique combinations of index and column values, and you want to reshape from long to wide format.

### 4.1 Basic pivot example

In [ ]:
# Load medal data
medals = pd.read_csv("table1.csv")
medals.head()

In [ ]:
# Pivot: Country as index, Medal types as columns
medals_wide = medals.pivot(
    index="Country", 
    columns="Medal", 
    values="Count"
)
medals_wide.head()

### 4.2 Understanding the pivot result

Let's examine one country to see how pivot works:

In [ ]:
# Argentina's medal data before pivot
medals[medals["Country"] == "ARG"]

In [ ]:
# Argentina's medal data after pivot
medals_wide.loc["ARG"]

### 4.3 Handling missing values

💡 **Tip**: Pivot creates NaN values when a combination doesn't exist. Fill them with 0 for counts.

In [ ]:
# Fill NaN values with 0
medals_wide.fillna(0).head()

### 4.4 Alternative: groupby + unstack

The same result can be achieved with groupby and unstack:

In [ ]:
# Equivalent using groupby + unstack
medals.groupby(["Country", "Medal"])["Count"].sum().unstack().head()

### 4.5 Limits of pivot

⚠️ **Warning**: `.pivot()` fails when you have duplicate combinations of index and column values.

Let's see an example:

In [ ]:
# Load data with year information
medals_year = pd.read_csv("table2.csv")

# Filter for Algeria
algeria = medals_year[medals_year["Country"] == "ALG"].copy()
algeria

Notice that Algeria won Bronze medals in multiple years, creating duplicate Country + Medal combinations.

In [ ]:
# ⚠️ INTENTIONAL ERROR - This will fail!
# algeria.pivot(index="Country", columns="Medal", values="Count")
# Error: ValueError - Index contains duplicate entries, cannot reshape

print("This would raise: ValueError - Index contains duplicate entries")

✅ **Key Takeaway**: Use `.pivot_table()` instead when you have duplicates or need aggregation.

## 5. Pivot table

📚 **Concept**: Pivot tables are more powerful than pivot - they handle duplicates through aggregation.

**When to use**:
- When `.pivot()` fails due to duplicate combinations
- When you need aggregation (sum, mean, count, etc.)
- When you want behavior similar to Excel pivot tables

### 5.1 Basic pivot table with aggregation

In [ ]:
# Aggregate duplicates with sum
algeria.pivot_table(
    index="Country", 
    columns="Medal", 
    values="Count", 
    aggfunc="sum"
)

### 5.2 Using multiple index columns

In [ ]:
# Keep year information by using it as an index
algeria.pivot_table(
    index=["Country", "Year"], 
    columns="Medal", 
    values="Count", 
    aggfunc="sum"
)

### 5.3 Applying to full dataset

💡 **Tip**: Default `aggfunc` is `"mean"`, but for counts use `"sum"`.

In [ ]:
# Pivot table on full dataset
medals_year.pivot_table(
    index="Country", 
    columns="Medal", 
    values="Count", 
    aggfunc="sum"
).head()

### 5.4 Additional features: fill_value and margins

In [ ]:
# Replace NaN with 0 and add row/column totals
medals_year.pivot_table(
    index="Country", 
    columns="Medal", 
    values="Count", 
    aggfunc="sum",
    fill_value=0,
    margins=True
).head()

### 5.5 Alternative: groupby + unstack

In [ ]:
# Same result using groupby + unstack
medals_year.groupby(["Country", "Medal"])["Count"].sum().unstack().head()

## 6. Crosstab

📚 **Concept**: `pd.crosstab()` creates frequency tables and summary statistics.

**When to use**: When you want to count occurrences or calculate statistics across categories.

In [ ]:
# Reload Titanic data
titanic = pd.read_csv("titanic.csv")

### 6.1 Basic frequency table

In [ ]:
# Count passengers by sex and class
pd.crosstab(titanic["sex"], titanic["pclass"])

### 6.2 Crosstab with aggregation

In [ ]:
# Calculate mean fare by sex and class
pd.crosstab(
    index=titanic["sex"], 
    columns=titanic["pclass"], 
    values=titanic["fare"], 
    aggfunc="mean"
)

### 6.3 Alternative: groupby + unstack

In [ ]:
# Same result using groupby
titanic.groupby(["sex", "pclass"])["fare"].mean().unstack()

### 6.4 Crosstab with normalize

💡 **Tip**: Use `normalize=True` to get proportions instead of counts.

In [ ]:
# Calculate proportion of total fares
pd.crosstab(
    index=titanic["sex"], 
    columns=titanic["pclass"], 
    values=titanic["fare"], 
    aggfunc="sum", 
    normalize=True
)

This shows that females in 1st class paid about 34.8% of all fares on the Titanic.

### 6.5 Alternative without crosstab

In [ ]:
# Same calculation using method chaining
(
    titanic.groupby(["sex", "pclass"])["fare"]
    .sum()
    .div(titanic["fare"].sum())
    .unstack()
)

## 7. Melt

📚 **Concept**: Melt converts data from wide format to long format.

**When to use**: When you need to unpivot data for analysis or visualization.

### 7.1 Basic melt example

In [ ]:
# Load wide format medal data
medals_2012 = pd.read_csv("table_2012.csv")
medals_2012.head()

In [ ]:
# Convert from wide to long format
medals_long = medals_2012.melt(
    id_vars="Country", 
    value_vars=["Gold", "Silver", "Bronze"], 
    var_name="Medal",
    value_name="Count"
)
medals_long.head(10)

✅ **Key Takeaway**: Melt is the opposite of pivot - it takes columns and converts them to rows.

## Summary

### When to use each method:

| Method | Purpose | Best For |
|--------|---------|----------|
| `.transpose()` or `.T` | Swap rows and columns | Quick inspection, rarely for analysis |
| `.unstack()` | Create columns from index | After groupby operations |
| `.pivot()` | Reshape with unique combinations | Clean data with no duplicates |
| `.pivot_table()` | Reshape with aggregation | Data with duplicates, Excel-like summaries |
| `pd.crosstab()` | Frequency tables and summaries | Quick cross-tabulations |
| `.melt()` | Convert wide to long format | Preparing data for visualization/analysis |

### ✅ Key Takeaways:

1. Most reshaping can be done with `groupby() + unstack()` as an alternative
2. Use `.pivot_table()` instead of `.pivot()` when you have duplicates
3. Method chaining with `.fillna()` helps handle missing values
4. Long format is often better for analysis, wide format is better for reading

💡 **Tip**: When in doubt, start with `groupby()` and add `.unstack()` to reshape!